# MM4250 sweeps

One notebook for every measurement on the switch. Run the five **Setup** cells, then run one section by hand. Don't use *Run All*: it stops on purpose after Setup.

## First cooldown, in order

VNA port 1 on fridge line RF4, port 2 on line RF2, unless a step says otherwise. `FIRST_COOLDOWN.md` has the why and what to expect.

| When | Run | Need | Set in Session |
|---|---|---|---|
| Once, before anything | **A0** port map (what's on each switch channel) and `CHAIN`. If there are check standards: their names in `Sweeps/calibration/standards.json` (**C0**) | **Required**: nothing that moves the switch runs until the port map is filled in, and A4 refuses unknown standard names | |
| Before the switch is mounted | **C1** bench e-cal: switch on the bench, VNA port 1 straight to RFC, standards on their ports; paste its set id into `BENCH` (A0) | Optional: adds A5's "vs bench" score; the correction doesn't use it | |
| Warm, in the fridge, before pumpdown, **LNA on** at its 296 K bias (1.80 V / 45 mA, s/n 5858H report; unpowered, nothing comes back up line RF2) | **A4** + **A5** | Recommended: finds wiring or method problems while the fridge can still be opened | `temp_str = "295K"` |
| (same) | **A2** thru check, **A3** compression check, **C2** cal kit on every port (needs the kit) | Optional | `temp_str = "295K"` |
| Before cooling | **A6** (every channel open). Set the LNA to its **4 K bias** (1.20 V / 20 mA; LNF: never cool down on the room-temperature values) and update `CHAIN` in A0 | **Required** (cooling with a channel closed can leave it stuck closed) | |
| On the way down, 30-80 K (where the switch can freeze closed) | **A1** stuck check (stop if anything is stuck), then **A6** | **Required** | `temp_str` = the reading, e.g. `"60K"` |
| Base | **A3** compression check (about a minute) | Recommended | `temp_str = "10mK"` |
| Base | **A4** + **A5** | **Required**: the measurement | `temp_str = "10mK"` |
| Before warming up | **A6** | **Required** | |
| Any time afterwards, at a desk | **B1** + **B2** resonator Q fit | Recommended: reads saved sets only, no instruments (set `RES_SET` to the set id and `ideals` to NIST's 3K) | |

Shortest path: A0 -> (warm A4 + A5) -> A6 -> cool -> A1 + A6 at 30-80 K -> A4 + A5 at base -> A6 before warming; the Q fit afterwards. A stop at 3 K is probably skipped (plan from 2026-10-22); if you do stop there: A1, A3, then A4 + A5 with `temp_str = "3K"`, and A6 before cooling on.

## Sections

| Section | What | VNA correction |
|---|---|---|
| **A** Cooldown | port map, stuck check, thru check, compression check, e-cal sets on the DUT and check-standard channels (calibrated S11 at each connector, from S21), check-standard scores, ALL_OPEN before temperature changes | off (the code enforces it) |
| **B** Resonator | Q fit from A's sets, in its own database | n/a |
| **C** Check standards & definitions (warm) | the standards file, the bench e-cal reference, your own definitions from a cal kit on every port compared with NIST's | off |
| **D** Bench | VNA cabled straight to the switch, no fridge: D1 quick look, D2 (1-port), D3 (one cable, 1-port cal), D4 (2-port, cal and/or uncal) | your choice / 1-port cal / `CAL_SET` |
| **E** | browse and plot any saved runs | n/a |
| **Close** | back up the databases, open the switch, release both instruments | |

**Files it needs**, in this folder: `vna_measure.py`, `sweep_db.py`, `read_db.py`, `plots.py`, `ecal.py`, `duts.py`, NIST's definitions (`ideals_3K/`, `ideals_295K/`, see `LAB_SETUP.md`), and scikit-rf for B's fit.

**What it writes** (from cooldown 5, set in Setup's **Databases** cell): every switch run into the lab's switch database, `E:\QCoDeS\<SWITCH_PROJECT>\Cooldown_05\<SWITCH_PROJECT>_Cooldown_05_vna.db` on the DAQ (Touchstone files, only with `touchstone=True` or `export_touchstone`, and `db_backups/` go beside it). Section B: the lab's resonator database, `E:\QCoDeS\<RESONATOR_PROJECT>\Cooldown_05\<RESONATOR_PROJECT>_Cooldown_05_vna.db`. Beside this notebook, as before: section C's `Sweeps/calibration/` (its own `mm4250_calibration.db` for the bench e-cal and kit runs, `standards.json`, used across cooldowns), D3's `.s1p` files, figures when a plot is saved. Every run from before cooldown 5 stays in `mm4250_sweeps.db` beside this notebook (`code_db_path()`).

## Setup

### Imports

Finds this folder and the `drivers/` folder above it (prints both), and imports everything the sections use. Also makes every run record which VNA cal set was active, as `vna_cal_set`.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import numpy as np

# The drivers live in whichever repo root is above this notebook -- this
# folder if it's the mm4250-switch repo, or the framework repo if these
# files were copied into users/<name>/. Walk up until we find it.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))       # vna_measure / sweep_db live here

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))       # drivers/ live here
print(f"notebook folder: {here}")
print(f"drivers from:    {root}")

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from datetime import date

import vna_measure
from vna_measure import setup_sweep, sweep_settings, measure_s11, measure_2port, _select
from sweep_db import run_oneport_sweep, run_twoport_sweep, run_ecal_set, run_kit_set, run_bench_standards, _leave_switch_open, use_lab_databases
from plots import plot_measurement, plot_sweep, summarize
from read_db import list_runs, load_run, export_touchstone, save_touchstone, backup_db, load_standards, default_db_path, default_cal_db_path, code_db_path
import ecal
import duts

# Record the active cal set's name in every run's metadata (vna_cal_set).
vna_measure.VNA_STATE_QUERIES["cal_set"] = "SENS:CORR:CSET:ACT? NAME"

### Connect

Creates `ksvna` (Keysight P5004B) and `switch` (MM4250). Connecting the switch **forces `ALL_OPEN`**: re-running this mid-session resets the switch. If the switch won't connect, another kernel still holds it. If `import hid` fails, see *Debugging* at the bottom.

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

### Session

Set these once per session. Every section below files its runs under them. `temp_str` is a folder label; in the fridge, also pass the actual mixing-chamber reading to `run_ecal_set(mxc_temp_k=...)`.

In [ ]:
date_str = date.today().strftime("%Y%m%d")   # or type one, e.g. "20261015"
temp_str = "295K"            # "295K", "3K", "10mK", ...
switch_serials = "SN0077"    # the switch under test (e.g. "SN0077_SN0078" for two)
CAL_SET = "20260911_1MHz_10GHz_female"   # the VNA's saved 2-port cal, for D4 and C1 (bench) - NOT needed for fridge
print(f"{switch_serials} | {date_str} | {temp_str}")

### Databases

Where this session's runs go: the lab's database helper (`scripts/database_helper.py` in the framework, guide `docs/Database_Instructions.md`) makes one database per project, cooldown and `name`: `E:\QCoDeS\<project>\Cooldown_<NN>\<project>_Cooldown_<NN>_<name>.db`, folders made on first use. The switch's runs (A, D, E) go to `SWITCH_PROJECT`, section B's resonator copies and fits to `RESONATOR_PROJECT`, both with `name="vna"`. Touchstone files and `db_backups/` go beside each database, in the cooldown folder. Section C's calibration database stays beside this notebook (its bench runs are used across cooldowns). It prints the three paths: check them before measuring.

- **Each new cooldown: change `COOLDOWN_NUM`.** A0's `COOLDOWN` label follows it.
- **Run it again after a kernel restart** (the setting lasts one Python session). Don't add your own `initialise_or_create_database_at` line.
- The lab's guide: only one notebook can write to a `.db` at a time, so a second notebook recording at the same time (a fridge log, say) should use another `name` or project.
- Runs from before cooldown 5 are in `mm4250_sweeps.db` beside this notebook: pass `code_db_path()` as `db_path` to read them (`list_runs(code_db_path())`, `ecal.find_set("<set id>", code_db_path())`).
- On the laptop (no lab helper) comment out the `use_lab_databases` line: everything then goes beside this notebook, as before.

In [ ]:
COOLDOWN_NUM = 5                          # the fridge's cooldown number: change it every cooldown
SWITCH_PROJECT = "MM4250"                 # every switch run (A, D, E)
RESONATOR_PROJECT = "Project8_Resonator"  # section B: the resonator's copies and fits
DB_NAME = "vna"                           # the helper's name: <project>_Cooldown_05_vna.db

DBS = use_lab_databases(COOLDOWN_NUM, SWITCH_PROJECT, RESONATOR_PROJECT, name=DB_NAME)   

### Stop here

No need to run. This cell exists so that *Run All* ends after Setup. From here, run a section by hand.

In [ ]:
raise SystemExit("Setup done. Now run one section below by hand.")

## Check the VNA

Saved cal sets, current sweep settings, whether correction is on (`1`) and which cal set is active. Worth a look before any section.

In [ ]:
print("Saved cal sets:", ksvna.ask("SENS:CORR:CSET:CAT? NAME").strip())
sweep_settings(vna=ksvna)
print("Correction on:", ksvna.ask("SENS:CORR:STAT?").strip())
print("Active cal set:", ksvna.ask("SENS:CORR:CSET:ACT? NAME").strip())

## A. Cooldown: DUTs in the fridge (e-cal)

Calibrated S11 of whatever is on each switch channel (resonator, SNTJ, ...), at that channel's SMA connector. The switch's internal open/short/load plus a definition of what each one is (NIST's by default) remove everything between the VNA and the switch. The **check standards** (the shorts and load in `STD_PORTS`) are measured in the same A4 set as the DUTs and scored in A5: that's how you know how good the correction is. **The order is the table at the top.**

**Wiring** (`20261016_Wiring_Diagram_testrun.pdf`): VNA port 1 -> fridge line RF4 -> isolator -> circulator -> switch RFC. The reflection goes back into the circulator and up line RF2 through the LNA to VNA port 2, so the switch's reflection is **S21**. Fridge line RF3 goes to one switch channel (`THRU_CH`) for the thru check. Fridge line names are not switch channels. The sweep stays inside 510-610 MHz, the specified band of the Quinstar circulator and isolator (quote S-QUO4203 in the shared `Project 8/Amplifiers/`): outside it the reflection doesn't come back up.

**A0** What's on each channel, the sweep, the cooldown label and the fridge thermometer. Re-run after a kernel restart.

- **The port map**: `DUTS` (devices under test), `STD_PORTS` (check standards: the known shorts and load, measured in every A4 set beside the DUTs and scored in A5; names from the standards file, C0) and `THRU_CH` (the channel fridge line RF3 is wired to, for A2). Each switch channel 1-6 at most once across the three. They start as `None`: A1-A4, C1 and C2a refuse to run until all three are filled in, and stop if a channel is used twice. `STD_PORTS = {}` if there are no standards this cooldown.
- `BENCH`: C1's bench e-cal set id, pasted from its printout (e.g. `"20261027T101502.318"`). A5 then also compares each check standard with its bench result, matched by name. Until then A5 scores them against their ideal values only. (The older VNA-cal bench's `{name: run id}` still works here too.)
- `DUTS` names what's on each channel. Every key gets measured, and the name is saved on that channel's runs (`dut_label`), comes back with `ecal.find_set`, and shows in the plot legend as "RF1 (resonator)".
- `CHAIN`: what the VNA can't report about the chain, saved on every A4 run (and back from `ecal.find_set` as `"metadata"`): the LNA's drain voltage and current as set, and any attenuator added on VNA port 1 outside the fridge. Update it whenever the bias changes (warm vs 5 K values) and re-run A0. `None` = not recorded.
- `TEMP_READER` reads the fridge temperature for you, once per run: any function with no arguments that returns kelvin, e.g. `lambda: fridge.mxc_temperature()` once the fridge's driver is connected (the name depends on that driver). Leave it `None` to type `mxc_temp_k` in A4 instead. If it fails before a set starts, nothing moves; if it fails partway, that run is saved without a reading.

In [ ]:
# Port map for this cooldown: switch channel (1-6) -> what's on it. EDIT once decided; each channel once.
DUTS      = None   # e.g. {1: "resonator", 2: "SNTJ"}
STD_PORTS = None   # check standards, e.g. {3: "short_A", 4: "short_B", 5: "load_1"}: names in Sweeps/calibration/standards.json (C0); {} if none
THRU_CH   = None   # e.g. 6: the channel fridge line RF3 is wired to (A2)

DUT_SPAN = dict(start=510e6, stop=610e6, points=2001, if_bandwidth=1e3, power=-20)   # circulator + isolator band (Quinstar spec)
COOLDOWN = f"CD{COOLDOWN_NUM}"   # labels every set this cooldown (C2 uses the same one); the number is set in Setup: Databases
TEMP_READER = None          # e.g. lambda: fridge.mxc_temperature(); None = type mxc_temp_k in A4
BENCH = None                # after C1 (bench e-cal), paste its set id here, e.g. "20261027T101502.318"
CHAIN = dict(lna_vd_v=None,        # LNA drain voltage as set, V (sn5858H report: 1.80 at 296 K, 1.20 at 4 K)
             lna_id_ma=None,       # LNA drain current as read, mA (report: 45 at 296 K, 20 at 4 K)
             port1_atten_db=0)     # attenuator added on VNA port 1 outside the fridge, dB


def check_port_map():
    """Stop before anything moves if the port map isn't filled in or a channel is used twice."""
    if DUTS is None or STD_PORTS is None or THRU_CH is None:
        raise ValueError("A0: fill in the port map (DUTS, STD_PORTS, THRU_CH) and re-run A0 first")
    used = [*DUTS, *STD_PORTS, THRU_CH]
    bad = [ch for ch in used if not (isinstance(ch, int) and 1 <= ch <= 6)]
    if bad:
        raise ValueError(f"A0 port map: channels are the numbers 1-6, got {bad}")
    twice = sorted({ch for ch in used if used.count(ch) > 1})
    if twice:
        raise ValueError(f"A0 port map: {', '.join(f'RF{ch}' for ch in twice)} used more than once")


try:
    check_port_map()
    role = {THRU_CH: "line RF3 (thru, A2)"}
    role.update({ch: f"standard: {name}" for ch, name in STD_PORTS.items()})
    role.update({ch: f"DUT: {name}" for ch, name in DUTS.items()})
    for ch in range(1, 7):
        print(f"RF{ch}: {role.get(ch, 'bare')}")
except ValueError as e:
    print("Port map not ready:", e)
unset = [k for k, v in CHAIN.items() if v is None]
print("CHAIN:", CHAIN, f"  (not recorded: {', '.join(unset)})" if unset else "")


**A1** Stuck check (**plot and printout only, nothing is saved**), on the way down in the 30-80 K range (where the nitrogen inside can freeze a switch closed; Menlo/NIST, 2026-10-22), and first thing at any stop where you'll take e-cal sets. It closes each internal standard and each DUT channel once, with `ALL_OPEN` in between. Top plot: raw |S21| of every position (ALL_OPEN in grey). Bottom: each position's median difference from the first ALL_OPEN, in order. Every grey bar should be small and every blue one clearly bigger; compare bars with each other, not to a fixed number. With `TEMP_READER` set, the title shows the reading. A closed position about as small as the grey bars = stuck open. A grey bar that looks like the position before it = stuck closed. The chain's gain is still changing while the fridge cools, so compare neighbouring bars only. **If anything is stuck, stop**: warm up (switches recover at 30 to 80 K), see `FIRST_COOLDOWN.md` step 5. Run **A6** after it. `CHECK` is every channel in A0's port map (DUTs, standards and the line RF3 channel). VNA port 1 on line RF4, as for A4. If the check stops partway (a VNA error, an interrupt), it still opens every channel before it quits.

In [ ]:
check_port_map()
CHECK = sorted(set(DUTS) | set(STD_PORTS) | {THRU_CH})
print("checking", [f"RF{ch}" for ch in CHECK], "(plot only, nothing is saved)")
setup_sweep(vna=ksvna, **DUT_SPAN)
ksvna.write('SENS:CORR:CSET:DEAC')
plate = f"{TEMP_READER():.1f} K" if TEMP_READER is not None else temp_str   # the reading if there's a reader

seq = ["ALL_OPEN", "INTERNAL_SHORT", "ALL_OPEN", "INTERNAL_LOAD", "ALL_OPEN"]
for ch in CHECK:
    seq += [ch, "ALL_OPEN"]
traces = []
try:
    for pos in seq:                     # measure_2port turns the source off for each move
        if isinstance(pos, int):
            freq, data = measure_2port(pos, vna=ksvna, switch=switch)
        else:
            freq, data = measure_2port(state=pos, vna=ksvna, switch=switch)
        traces.append((pos, data["S21"]))
finally:
    _leave_switch_open(switch, ksvna)   # every channel open, even if the check dies partway

# How far each position is from the first ALL_OPEN (complex difference, so phase counts too)
ref = traces[0][1]
names = {**{ch: f"RF{ch} ({n})" for ch, n in DUTS.items()},
         **{ch: f"RF{ch} ({n})" for ch, n in STD_PORTS.items()},
         THRU_CH: f"RF{THRU_CH} (line RF3)"}
labels = [names.get(p, str(p)) for p, _ in traces]
diff = [np.median(np.abs(s21 - ref)) for _, s21 in traces]
for lab, d in zip(labels, diff):
    print(f"{lab:>24}:  median |S21 - first ALL_OPEN| = {d:.3g}")

import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
for (pos, s21), lab in zip(traces, labels):
    db = 20 * np.log10(np.abs(s21))
    if pos == "ALL_OPEN":
        ax1.plot(freq / 1e6, db, color="0.7", lw=1)
    else:
        ax1.plot(freq / 1e6, db, lw=1.3, label=lab)
ax1.plot([], [], color="0.7", lw=1, label="ALL_OPEN (every one)")
ax1.set_xlabel("Frequency (MHz)")
ax1.set_ylabel("|S21| raw (dB)")
ax1.set_title(f"A1 stuck check, {switch_serials}, {plate}: raw, not saved")
ax1.legend(fontsize=8, ncol=2)
ax2.bar(range(len(diff)), diff, color=["0.7" if p == "ALL_OPEN" else "C0" for p, _ in traces])
ax2.set_xticks(range(len(diff)))
ax2.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("median |S21 - first ALL_OPEN|")
ax2.set_title("Grey (ALL_OPEN) should all be small, blue all clearly bigger", fontsize=10)
fig.tight_layout()
plt.show()


**A2** Thru check through fridge line RF3: the direct test that the switch closes and opens. Line RF3 is wired to switch channel `THRU_CH`. With VNA port 1's cable moved from line RF4 to line RF3 at the top of the fridge (port 2 stays on line RF2), the signal goes line RF3 -> RF`THRU_CH` -> RFC -> circulator -> line RF2, so S21 is a transmission: high with RF`THRU_CH` closed, down at the switch's isolation with it open. The cell closes and opens that channel twice and prints the gap.

- It tests that one channel (and RFC); A1 tests the others by reflection. Run it **warm once** (switch mounted, before pumpdown) to see what "good" looks like, and **cold right after A1**.
- Saved to the main database as setup `thru_lineRF3`. It leaves the switch `ALL_OPEN`, even if it fails.
- **Move port 1 back to line RF4 afterwards**: A1, A4 and C2 need it there.
- The 20 dB pass mark is my guess (well under the switch's isolation); the warm run tells you the real closed/open levels.
- Line RF3 has less attenuation than line RF4 (28 dB vs 36 dB plus the Eccosorb filter, per the 2026-10-16 wiring diagram), so with the thru closed more power reaches the LNA than in any A4 run (still under its compression point, my estimate).

In [ ]:
check_port_map()
temp = temp_str                     # this temperature's label, e.g. "295K", "3K"
input("Move VNA port 1's cable from line RF4 to line RF3 (port 2 stays on line RF2), then press Enter")
setup_sweep(vna=ksvna, **DUT_SPAN)
ksvna.write('SENS:CORR:CSET:DEAC')

seq = ["ALL_OPEN", THRU_CH, "ALL_OPEN", THRU_CH, "ALL_OPEN"]   # close and open the thru channel twice
try:
    runs = run_twoport_sweep(seq, date_str, temp, switch_serials, setup="thru_lineRF3",
                             labels={THRU_CH: "line RF3 thru"}, temp_reader=TEMP_READER,
                             vna=ksvna, switch=switch)
finally:
    _leave_switch_open(switch, ksvna)   # every channel open, even if it dies partway

level = []
for pos, r in zip(seq, runs):
    level.append(20 * np.log10(np.median(np.abs(load_run(r)[1]["S21"]))))
    print(f"{str(pos):>9}  run {r}:  median |S21| = {level[-1]:6.1f} dB")
gap = min(l for p, l in zip(seq, level) if p == THRU_CH) - max(l for p, l in zip(seq, level) if p == "ALL_OPEN")
print(f"\nRF{THRU_CH} closed vs ALL_OPEN, worst case: {gap:.1f} dB apart")
print("OK: it closes and opens" if gap > 20 else
      "LOOK AT THIS: closed and open are close together. Stuck open/closed, or port 1 isn't on line RF3?")
print("Now move VNA port 1 back to line RF4.")

**A3** Compression check, with the LNA on, before the first A4 at each temperature (warm in the fridge and at base). S-parameters are ratios, so if the amplifier chain is linear the internal short's |S21| is the same at `DUT_SPAN`'s power and 10 dB below it. If it drops at the higher power, the chain is compressed (not expected with one LNA: roughly 15-20 dB of margin at -20 dBm, my estimate from the 2026-10-16 wiring) and the e-cal would be off: lower `power` in `DUT_SPAN` (A0) until this passes. Nothing is saved; the switch ends `ALL_OPEN`. The 0.2 dB pass mark is my guess (NIST's accuracy is about 0.1 dB); at the lower power the trace is noisier, so a small wobble either way is noise, not compression.

In [ ]:
check_port_map()
P = DUT_SPAN["power"]
setup_sweep(vna=ksvna, **DUT_SPAN)
ksvna.write('SENS:CORR:CSET:DEAC')
level = {}
try:
    for p in (P, P - 10):
        setup_sweep(power=p, vna=ksvna)
        freq, data = measure_2port(state="INTERNAL_SHORT", vna=ksvna, switch=switch)   # source off for the move
        level[p] = 20 * np.log10(np.median(np.abs(data["S21"])))
        print(f"{p:6.1f} dBm:  internal short, median |S21| = {level[p]:7.2f} dB")
finally:
    setup_sweep(power=P, vna=ksvna)
    _leave_switch_open(switch, ksvna)

change = level[P] - level[P - 10]
print(f"\n|S21| at {P} dBm minus at {P - 10} dBm: {change:+.2f} dB")
print("OK: linear" if change > -0.2 else
      f"COMPRESSED: lower DUT_SPAN['power'] in A0 (try {P - 10}), re-run A0, and run this again")

**A4** One e-cal set: the internal standards, every DUT and check-standard channel, then the standards again (the drift check), `repeats` times. Each position is a 2-port run, so all four S-parameters are saved. It leaves the switch `ALL_OPEN`, even if it fails. Don't change `setup_sweep` partway through a set. `note` is for things about the whole set (fridge state, wiring, why you took it).

- `temp` labels the set ("295K", "3K", "10mK", ...); `mxc_temp_k` is the actual reading of the plate the switch is on. Pass that or `TEMP_READER`, not both.
- **Check standards**: every channel in `STD_PORTS` is measured in the same set and tagged with its standards-file entry (type, part, serial, offset), so it goes through exactly the same correction as the DUTs. Needs `Sweeps/calibration/standards.json` (C0) to have their names; an unknown name stops the set before anything moves.
- VNA correction has to be off. **With the VNA's own cal on underneath** (optional, warm): activate it instead and pass `vna_cal=True`. Without that flag A4 refuses to run with a cal on, and with it A4 refuses to run with the cal off. The e-cal result should come out the same either way; what changes is the grey trace in A5, which becomes "VNA cal only" (cables removed, circulator and switch still in). `setup_sweep` then has to match the cal's settings (`CAL_SET`: 1 MHz to 10 GHz, 10000 points, 1 kHz), or the VNA interpolates the cal or turns it off.
- Power: what reaches the switch is this minus the input line's attenuation. NIST had about -52 dBm at the switch. For less noise, lower `if_bandwidth` first; a slower set gives the chain longer to drift, so check the drift in A5 when you do.

In [ ]:
temp = temp_str                               # "295K" warm, "3K", then the base label, e.g. "10mK"

check_port_map()
setup_sweep(vna=ksvna, **DUT_SPAN)
ksvna.write('SENS:CORR:CSET:DEAC')            # e-cal wants raw data
# ksvna.write(f'SENS:CORR:CSET:ACT "{CAL_SET}",0')   # or the VNA's cal on underneath: also set vna_cal=True

cal = run_ecal_set(date_str, temp, switch_serials,
                   channels=sorted(set(DUTS) | set(STD_PORTS)),   # DUTs and check standards, one set
                   labels=DUTS,
                   cal_standards=STD_PORTS or None,   # tags each check standard with its standards-file entry
                   db_path=default_db_path(),         # the main database, beside the DUT runs
                   repeats=2,
                   n_ports=2,                 # all four S-params at every position
                   vna_cal=False,             # True = the VNA's own cal on underneath (see above)
                   mxc_temp_k=None,           # the plate's reading, e.g. 3.1 (leave None with TEMP_READER)
                   temp_reader=TEMP_READER,
                   cooldown=COOLDOWN,
                   metadata=CHAIN,            # LNA bias, outside attenuation (A0)
                   note="",                   # e.g. "in fridge, warm, before pumpdown"
                   vna=ksvna, switch=switch)

**A5** Corrects the set and shows it, in one call: the drift check, the correction, the spread between repeats, and the plot. Then scores the check standards.

- Definitions: picked from `temp_str` by `ecal.nist_definitions_for`: NIST's **295K** at room temperature, their **3K** at 4.5 K and below (NIST found those hold to 25 mK). In between (e.g. "60K") there are none: it warns and uses 3K, and that result is rough. Warm, your own from C2 (`my_ideals`) also work, or `"perfect"` ones (open +1, short -1, load 0; no data needed, but the result then still includes the switch's path to the channel).
- What to look at: the drift's **calibrated** line (how far the standards moved during the set, in reflection units at the switch; NIST's same-temperature repeatability was a few 1e-2; well above that means take it again), the spread between repeats, then the plot. The raw line under it is scaled by the chain's loss and gain, so don't compare that one with NIST.
- **Check standards** (`ecal.check_standards`): each short and load in `STD_PORTS`, scored against what it should read (short -1, load 0) and, once `BENCH` is filled in (C1), against the same standard's bench e-cal result: warm in the fridge that difference should be about zero (the e-cal removes the fridge path); cold it shows what cooling changed. The bench went through NIST's 295 K definitions too, so it isn't an independent truth: -1 and 0 stay the absolute check. The worst number is the correction's error at that port's connector. For scale (NIST's own switch, 510-610 MHz, my calculation from their files): the same port's 3 K definitions from two of their cooldowns differ by up to 0.02-0.04 for a short and 0.01-0.035 for a load; one port's definitions used on another, up to 0.028 / 0.047; warm, two cooldowns, about 0.01. Much bigger than that cold means NIST's definitions fit your switch worse than their own switch varies between cooldowns. A load's cold value isn't known, so its number is an upper limit. Each standard checks its own port; C2 is the warm check that the DUT ports behave the same.
- A set from an earlier session: `ecal.list_sets()` shows them all, and `cal = ecal.find_set("<set id>")` loads one. From before cooldown 5: `ecal.list_sets(code_db_path())`, `ecal.find_set("<set id>", code_db_path())`.
- `ecal.export_corrected(result)` writes one .s1p per channel, for anything outside this notebook (the Q fit, for one).

In [ ]:
ideals = ecal.find_ideals(ecal.nist_definitions_for(temp_str), start=here)   # NIST's, for this temperature
result = ecal.correct_and_plot(cal, ideals, sparam="S21", show_raw=True)
bench = BENCH                                                   # C1's {name: run id} (old VNA-cal bench), or:
if isinstance(BENCH, str):                                      # C1's bench e-cal set id: correct it the same way
    bench = ecal.correct_set(ecal.find_set(BENCH, default_cal_db_path()), ecal.find_ideals("295K", start=here),
                             sparam="S11", db_path=default_cal_db_path())
checks = ecal.check_standards(result, bench)                    # the check standards: how good is the correction?
# result = ecal.correct_and_plot(cal, "perfect", sparam="S21", show_raw=True)
# result = ecal.correct_and_plot(cal, my_ideals, sparam="S21")   # your own, from C2 (warm only)

# ecal.export_corrected(result)                                  # .s1p per channel

**A6** Before any temperature change (cooling to base, warming up): the switch has to be `ALL_OPEN`. Cooling with a channel closed can leave it stuck closed (Menlo APN-0021). A1 and A4 already leave it open; this makes sure, with the source off for the move. Board LEDs: only D5 (the always-on status LED) should be lit. The printout is the state last sent, not read back: the lab driver's readback lags (CLAUDE.md), so the LEDs are the check.

In [ ]:
_select(state="ALL_OPEN", switch=switch, vna=ksvna)    # source off while it moves
print("switch, last sent:", switch.state.cache.get(get_if_invalid=False))   # not read back: the lab driver's readback lags

## B. Resonator

The resonator on its switch channel, from A's e-cal sets: Q fit, plots, and its **own database**, the lab's `RESONATOR_PROJECT` database for this cooldown (Setup: Databases), so it isn't mixed in with the switch's data. The code is `duts.py`; the fit needs scikit-rf.

- **When:** after A4 + A5, once per temperature (warm, 3 K, base). B1 needs A5's `ideals` (or set them here: `ecal.find_ideals("3K", start=here)`, `"295K"` warm).
- **What goes in the database:** B1 copies the set's raw runs for the resonator into it (the switch's internal standards and the resonator channel, every repeat; not the SNTJ or other channels), with all their metadata. The main database keeps its own copy. B2 adds the fit. Everything about the resonator can then be redone from that one file.
- **The fit** is `q_fit_resonator_20261008.ipynb`'s: scikit-rf Qfactor, reflection, NLQFIT7, window 2 f_L/Q_L. The starting guesses come from the data, so it works cold too. If it warns about too few points in the fit window (a narrow resonance cold), take another A4 set with a narrow `DUT_SPAN` around the resonance.
- **Channel:** the one `DUTS` calls "resonator". Set `RES_CHANNEL` if it's labelled something else.
- **An older set:** `RES_SET = "<set id>"` (`ecal.list_sets()` lists them), with `ideals` for its temperature. A set from before cooldown 5 is in the old database: also pass `main_db_path=code_db_path()` to `copy_set` (it is copied into this cooldown's resonator database). Checked: the 2026-10-08 set (`"20261008T160234.969"`, NIST 295K) gives Q_L 39.9, Q_0 73.0, Q_e 88.1, beta 0.83, the same as the Q notebook.

**B1** Copy, correct, fit, plot. Nothing is saved as a fit until B2. Figures: `figures/<serials>/<date>/<temp>/resonator_q_fit_RF<n>_<set>` (.pdf and .png), the one-page fit and ____-style data vs fit.

In [ ]:
RES_SET = cal                    # this temperature's A4 set, or a set id, e.g. "20261008T160234.969"
RES_CHANNEL = None               # None = the channel DUTS calls "resonator"

res_set = duts.copy_set(RES_SET, channel=RES_CHANNEL)    # raw runs -> the resonator database (skips any already there); before cooldown 5: main_db_path=code_db_path()
res = duts.correct_resonator(res_set, ideals)             # A5's definitions, on the resonator db's copy
fit = duts.fit_resonator(res, channel=RES_CHANNEL)
duts.plot_resonator_fit(fit, save=True)
duts.plot_resonator_detail(fit, save=True);

**B2** Save the fit into the resonator database (the corrected trace plus f0, Q_L, Q_0, Q_e, beta, where it came from, and the fridge reading if A4 recorded one). It refuses to save the same set + definitions + fit settings twice; `allow_duplicate=True` keeps both. `note` is for anything about this fit.

In [ ]:
duts.save_resonator(fit, note="")

**B3** Everything saved so far, any time (no instruments needed): the table, then f0, Q_0, Q_e and beta against temperature (the fridge reading where A4 recorded one, else the set's temp label). Saved as `figures/<serials>/resonator_qfit_vs_temp`. To refit a saved one: `freq, gamma, row = duts.load_resonator(run_id)`, then `duts.fit_resonator((freq, gamma), label="...")`.

In [ ]:
duts.list_resonator()
fig, rows = duts.plot_resonator_vs_temp(save=True)

## C. Check standards and definitions (warm groundwork)

The check standards (the shorts and load in `STD_PORTS`) are measured in every **A4** set, beside the DUTs, and **A5** scores them: that's the verification, at every temperature. This section is the warm groundwork for it, all into the calibration database, `Sweeps/calibration/mm4250_calibration.db`:

- **C0** the standards file: what each check standard is.
- **C1** bench e-cal, before the switch is mounted: the switch on the bench with the check standards on their ports, calibrated by its own internal standards, so A5 can compare each standard in the fridge with the same standard on the bench.
- **C2** your own definitions from a cal kit on every port (warm, before the fridge is closed), compared with NIST's port by port. The only direct look at how far NIST's switch is from yours on the **DUT** ports: the check standards can only see their own ports.

Not done this cooldown: building your own cold definitions NIST-style (3+ different standards per port, rotated across cooldowns). `ecal.list_standard_runs()` and `ecal.standards_coverage()` still list what's been measured if that comes back.

**C0** The standards file, once: copy `standards_example.json` (this folder) to `Sweeps/calibration/standards.json` and fill in the check standards you actually have (type `short`/`load`/`open`/`other`, part, serial, `offset_mm` 0 for a flush short). Names in `STD_PORTS` (A0) must match it; A4 and C1 stop before anything moves if one doesn't. One name per physical part, e.g. `short_A` and `short_B`.

In [ ]:
standards = load_standards()          # Sweeps/calibration/standards.json
for name, s in standards.items():
    print(f"{name:20s} {s['type']:6s} {s.get('description', '')}")

if globals().get("STD_PORTS"):              # set in A0's port map
    missing = sorted(set(STD_PORTS.values()) - set(standards))
    print("not in standards.json yet (A4 and C1 will refuse):", missing if missing else "none")

**C1** Bench e-cal, 295 K, before the switch is mounted (the VNA's cal kit isn't used: Charlie, 2026-10-27). The switch on the bench, VNA port 1 straight to RFC with a short cable, the check standards (and any DUTs already on the switch) on their ports exactly as in A0's port map. VNA correction **off**: the switch's internal open/short/load do the calibration, as in A4, but 1-port (S11), since there's no circulator. Same sweep as A4 (`DUT_SPAN`), so it covers A5's band. Saved to the calibration database (`cooldown="bench"`), corrected with NIST's 295 K definitions and scored against -1 / 0 right here.

- **Paste the printed set id into `BENCH` in A0.** A5 then compares each check standard with its bench result, matched by name.
- What the comparison means: the bench and the fridge sets use the same switch, the same connectors and the same definitions, so warm in the fridge the difference should be about zero (that's the test that the e-cal removes cables, circulator and LNA). Cold, it shows what cooling changed. It isn't an independent truth (it went through NIST's definitions too); -1 and 0 stay the absolute check.
- The VNA-cal version (`run_bench_standards` with `CAL_SET`) is still in `sweep_db`; `BENCH` accepts its `{name: run id}` too.


In [ ]:
check_port_map()
BENCH_CHANNELS = sorted(set(DUTS) | set(STD_PORTS))   # whatever is on the switch on the bench: drop DUTs that aren't on yet
input("Bench: VNA port 1 straight to the switch's RFC, standards (and DUTs) on their ports as in A0. Press Enter")
setup_sweep(vna=ksvna, **DUT_SPAN)            # same sweep as A4, so the bench covers A5's band
ksvna.write('SENS:CORR:CSET:DEAC')            # raw: the switch's own standards do the calibration

bench_cal = run_ecal_set(date_str, "295K", switch_serials,
                         channels=BENCH_CHANNELS,
                         labels={ch: n for ch, n in DUTS.items() if ch in BENCH_CHANNELS},
                         cal_standards=STD_PORTS,          # tags the standards; sends the set to the calibration db
                         db_path=default_cal_db_path(),
                         cooldown="bench",
                         n_ports=1,                        # bench: the reflection is S11 (no circulator)
                         repeats=2,
                         note="bench e-cal reference, no fridge",
                         vna=ksvna, switch=switch)
bench_res = ecal.correct_and_plot(bench_cal, ecal.find_ideals("295K", start=here), sparam="S11",
                                  show_raw=True, db_path=default_cal_db_path())
ecal.check_standards(bench_res)                            # on the bench, against -1 / 0
print(f'\nBENCH = "{bench_cal["set"]}"    <- paste into A0')


### C2. Your own definitions from a cal kit, every port (warm only)

**C2a** measures the internal standards, then asks you to put the kit's open, short and load on each port's RF connector in turn (every channel is opened before each prompt), then the internal standards again. `ecal.make_ideals` turns that into a folder of definitions, `Sweeps/calibration/ideals_295K_<serials>_kit<set id>/`, which works anywhere NIST's do.

- Kit every port with something on it, DUTs **and** check standards (`KIT_PORTS`): comparing the two kinds is the point. Add `THRU_CH` if you can reach its connector.
- Whatever is normally on the port comes off for C2a and goes back on afterwards. The definitions belong to the switch alone, so cables or the circulator can move in between.
- VNA correction off (or on with `vna_cal=True`, as in A4). Same span as A4 (`DUT_SPAN`).
- `kit_defs=None` assumes a perfect kit, which is fine at these frequencies. See `help(ecal.make_ideals)` for giving it the kit's real definitions.

**C2b** How far NIST's 295 K definitions are from yours, port by port: the error a short and a load would read with on each port if you correct with NIST's (`ecal.compare_definitions`). **Compare the DUT ports with the check-standard ports**: if they're about the same size, the cold check-standard scores in A5 are a fair stand-in for the DUT ports. For scale, NIST's own switch between two warm cooldowns: up to about 0.01. Then a warm A4 set (`cal`), first DUT channel, corrected three ways: NIST's, yours, perfect.

In [ ]:
check_port_map()
KIT_PORTS = sorted(set(DUTS) | set(STD_PORTS))   # every port with something on it (add THRU_CH if reachable)
setup_sweep(vna=ksvna, **DUT_SPAN)
ksvna.write('SENS:CORR:CSET:DEAC')            # raw

kit = run_kit_set(date_str, "295K", switch_serials,
                  channels=KIT_PORTS,         # each prompts for open, short, load
                  note="",                    # e.g. which kit
                  db_path=default_cal_db_path(),
                  vna=ksvna, switch=switch)
my_ideals = ecal.make_ideals(kit, db_path=default_cal_db_path())   # -> Sweeps/calibration/ideals_295K_<serials>_kit<set id>/

# kit = ecal.find_kit_set("<set id>", db_path=default_cal_db_path())   # fresh kernel; ecal.list_kit_sets(default_cal_db_path()) lists them
# my_ideals = default_cal_db_path().parent / "ideals_295K_SN0077_kit<set id>"   # a folder made earlier

In [ ]:
nist = ecal.find_ideals("295K", start=here)
freq = np.linspace(DUT_SPAN["start"], DUT_SPAN["stop"], 201)
kit_ports = sorted(set(DUTS) | set(STD_PORTS))
diff = ecal.compare_definitions(my_ideals, nist, freq, ports=kit_ports)   # NIST's used where yours are right
print("DUT ports:", sorted(DUTS), " check-standard ports:", sorted(STD_PORTS))

ch = sorted(DUTS)[0]                                             # a warm A4 set (`cal`), three ways
results = [ecal.correct_set(cal, nist, sparam="S21"),
           ecal.correct_set(cal, my_ideals, sparam="S21"),
           ecal.correct_set(cal, "perfect", sparam="S21")]
ecal.plot_compare(results, labels=["NIST 295K", f"{switch_serials} (kit)", "perfect"],
                  channel=ch, show_raw=True)                     # save="compare.png" for a file

## D. Bench: VNA cabled straight to the switch (no fridge)

Everything in this section assumes the VNA's cables go straight to the switch (or to one device), not through the fridge's circulator. In the fridge wiring these give meaningless numbers. Each cell sets the VNA up itself.

### D1. Quick look (nothing saved)

One position, measured with whatever the VNA is currently set to. `measure_s11(3)` or `measure_s11(state="INTERNAL_LOAD")`; `measure_2port(3)` for all four S-parameters. Every line is commented out on purpose, because it moves the switch: uncomment one to use it, and never while the fridge is changing temperature (`FIRST_COOLDOWN.md` step 4; the only exception is A1 in step 5).

In [ ]:
# freq, s11 = measure_s11(3, vna=ksvna, switch=switch)
# summarize(freq, s11, title="RF3")
# ax = plot_measurement(freq, s11, title="RF3")

# freq, data = measure_2port(3, vna=ksvna, switch=switch)     # moves the switch to RF3
# ax = plot_measurement(freq, data, title="RF3", phase=True)

### D2. 1-port sweep

S11 at each position in `positions`: channel numbers, state names (`"ALL_OPEN"`, `"INTERNAL_LOAD"`, `"INTERNAL_SHORT"`, `"RFC_RF1"`...), or a mix. Raw by default; to have the VNA correct it, activate `CAL_SET` instead (a 2-port cal covers port 1's S11 too), with D4's `setup_sweep` (CAL_SET's settings).

In [ ]:
setup_sweep(start=1e9, stop=10e9, points=1001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')            # raw

positions = [1]
setup = "RF1"                                  # names the cabling, e.g. "RF3" -> folder RF3_uncal

runs = run_oneport_sweep(positions, date_str, temp_str, switch_serials, setup=setup,
                         vna=ksvna, switch=switch)      # touchstone=True for .s1p files too
ax = plot_sweep(runs)

### D3. Calibrated S11: one cable + the VNA's own 1-port cal (no switch)

First do a 1-port Smart Cal on the VNA with your cal kit (Cal -> Smart Cal -> 1-Port) and save the cal set. Put its name in `ONEPORT_CAL` and say what's on the cable in `NAME` (it names the file). Don't run `setup_sweep` first: `,0` keeps the VNA's current settings, which must be the ones the cal was taken at.

In [ ]:
ONEPORT_CAL = "your_cal_set_name"   # EDIT: the cal set you saved after Smart Cal, not the kit name
NAME = "resonator"                  # EDIT: what's on the cable; names the file

if ONEPORT_CAL == "your_cal_set_name":
    raise ValueError("D3: put your saved 1-port cal set's name in ONEPORT_CAL")
out = here / "Sweeps" / "no_switch" / f"{NAME}_{date_str}_{temp_str}.s1p"
if out.exists():
    raise FileExistsError(f"{out.name} already exists: change NAME")

ksvna.write(f'SENS:CORR:CSET:ACT "{ONEPORT_CAL}",0')     # VNA's 1-port cal on, current settings kept
freq, s11 = measure_s11(vna=ksvna)                       # no channel: the switch isn't touched
ax = plot_measurement(freq, s11, phase=True, title=NAME)

save_touchstone(freq, s11, out)
print("saved", out)

### D4. 2-port sweep: cal, uncal, or both

VNA port 1 on RFC, port 2 on RF`n`. S11, S12, S21, S22 at each position. `MODE = "both"` measures the same positions with `CAL_SET` on, then off, so the two can be compared directly; `"cal"` or `"uncal"` does just one. Measuring `[n, "ALL_OPEN"]` gives isolation without touching a cable. The sweep is `CAL_SET`'s own settings (1 MHz to 10 GHz, 10000 points, 1 kHz) so the VNA doesn't interpolate the cal. Change `n`, re-cable, run again.

In [ ]:
n = 1                    # EDIT: the RF channel VNA port 2 is cabled to
MODE = "both"            # "cal", "uncal", or "both"
positions = [n, "ALL_OPEN", "INTERNAL_LOAD", "INTERNAL_SHORT"]

if MODE not in ("cal", "uncal", "both"):
    raise ValueError('MODE is "cal", "uncal" or "both"')
setup_sweep(start=1e6, stop=10e9, points=10000, if_bandwidth=1e3, power=-20, vna=ksvna)   # CAL_SET's own settings
runs = []
if MODE in ("cal", "both"):
    ksvna.write(f'SENS:CORR:CSET:ACT "{CAL_SET}",0')     # ,0 keeps the settings above -> RF<n>_cal
    runs += run_twoport_sweep(positions, date_str, temp_str, switch_serials,
                              setup=f"RF{n}", vna=ksvna, switch=switch)
if MODE in ("uncal", "both"):
    ksvna.write('SENS:CORR:CSET:DEAC')                   # -> RF<n>_uncal
    runs += run_twoport_sweep(positions, date_str, temp_str, switch_serials,
                              setup=f"RF{n}", vna=ksvna, switch=switch)
print("runs", runs)
ax = plot_sweep(runs, sparam="S21")

## E. Browse and plot

`plot_sweep` reads from the database, so any runs from any session work: run ids, or a folder of Touchstone files. `list_runs()` shows what's there.

In [ ]:
for r in list_runs()[-15:]:                 # the newest runs; list_runs() has them all; list_runs(code_db_path()) = before cooldown 5
    print(r["run_id"], r["experiment"], r["name"], r["dut_label"] or "")

RUNS = []                                   # EDIT: run ids from the list above, e.g. [25, 26, 27]
if RUNS:
    ax = plot_sweep(RUNS, sparam="S21")
# ax = plot_measurement(run=RUNS[0], phase=True)                          # one run, magnitude and phase
# ax = plot_sweep("Sweeps/<serials>/<date>/<temp>/<setup>_cal", sparam="S11")   # a folder of Touchstone files
# export_touchstone(RUNS)                                                 # .s1p/.s2p files

## Close

Backs up the database (and section C's and B's, if there are any) to `db_backups/` (nothing is written if no runs are new since the last backup), then turns the VNA output off, opens every switch channel and releases both instruments. Run it before another notebook needs the switch, because the MM4250's USB connection is exclusive.

In [ ]:
backup_db()          # snapshot to db_backups/ beside the .db; skipped if nothing new
if default_cal_db_path().is_file():
    backup_db(default_cal_db_path())   # section C's database, same rule
if duts.resonator_db_path().is_file():
    backup_db(duts.resonator_db_path())   # section B's database, same rule

ksvna.output(False)
switch.open_all()
ksvna.close()
switch.close()

## Debugging: hidapi

Only needed if `import hid` fails. See the hidapi section of `LAB_SETUP.md`.

In [ ]:
#checking hidapi properly active
import sys, subprocess, importlib
r = subprocess.run([sys.executable, "-m", "pip", "install", "hidapi"], capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])

importlib.invalidate_caches()
try:
    import hid
    print("OK:", hid.__file__)
except Exception as e:
    print("FAILED:", type(e).__name__, e)
print("python:", sys.executable)